In [1]:
# NLP + K-Means Clustering for `publishers.csv`

## 1. Import Libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
```

---

## 2. Load the Dataset

```python
# Load publishers.csv
df = pd.read_csv("publishers.csv")

# Display the first 5 rows
df.head()
```

Check the dataset:

```python
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())
```

---

## 3. Inspect the Dataset

```python
df.info()
```

```python
df.describe(include="all")
```

---

## 4. Download NLP Resources

Run this cell once:

```python
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
```

---

## 5. Identify Text Columns

First, display the columns containing text:

```python
text_columns = df.select_dtypes(include=["object"]).columns

print("Text columns:")
print(list(text_columns))
```

If, for example, your dataset contains columns such as:

* `publisher`
* `description`
* `category`

you can combine them.

### Option A — Automatically combine all text columns

```python
text_columns = df.select_dtypes(include=["object"]).columns

df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

df[["combined_text"]].head()
```

### Option B — Select specific columns

If you know which columns contain useful text, use:

```python
text_columns = ["publisher", "description", "category"]

df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)
```

**Change the column names above to match your `publishers.csv`.**

---

## 6. Clean the Text

```python
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation and numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and short words
    words = [
        word for word in words
        if word not in stop_words and len(word) > 2
    ]

    # Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)
```

Apply the cleaning:

```python
df["clean_text"] = df["combined_text"].apply(clean_text)

df[["combined_text", "clean_text"]].head()
```

---

## 7. Convert Text into TF-IDF Features

TF-IDF converts the publisher text into numerical features that K-Means can use.

```python
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X.shape)
```

---

## 8. Find a Suitable Number of Clusters

We can test different values of `k`.

```python
inertias = []
silhouette_scores = []

k_values = range(2, 11)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, labels))
```

---

## 9. Plot the Elbow Method

```python
plt.figure(figsize=(8, 5))

plt.plot(k_values, inertias, marker="o")

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.show()
```

Look for the point where the decrease in inertia starts to slow down. This gives a possible value for `k`.

---

## 10. Plot Silhouette Scores

```python
plt.figure(figsize=(8, 5))

plt.plot(k_values, silhouette_scores, marker="o")

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different Values of k")

plt.show()
```

A higher silhouette score generally indicates better separation between clusters. Use the plots together with knowledge of your dataset to choose `k`.

---

## 11. Apply K-Means

For example, if you choose 4 clusters:

```python
k = 4

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

df.head()
```

---

## 12. Count Publishers in Each Cluster

```python
cluster_counts = df["cluster"].value_counts().sort_index()

print(cluster_counts)
```

Visualize them:

```python
plt.figure(figsize=(8, 5))

cluster_counts.plot(kind="bar")

plt.xlabel("Cluster")
plt.ylabel("Number of Publishers")
plt.title("Number of Publishers per Cluster")

plt.show()
```

---

## 13. Find Important Words in Each Cluster

This helps us understand what each cluster represents.

```python
terms = vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster in range(k):
    print(f"\nCluster {cluster}")
    print("-" * 50)

    top_words = [
        terms[ind]
        for ind in order_centroids[cluster, :15]
    ]

    print(", ".join(top_words))
```

---

## 14. Display Publishers in Each Cluster

```python
for cluster in sorted(df["cluster"].unique()):

    print("\n" + "=" * 60)
    print(f"CLUSTER {cluster}")
    print("=" * 60)

    cluster_data = df[df["cluster"] == cluster]

    display(cluster_data.head(20))
```

---

## 15. Reduce Dimensions for Visualization

TF-IDF can contain thousands of dimensions, so PCA can be used to create a 2D representation.

```python
pca = PCA(n_components=2, random_state=42)

X_dense = X.toarray()

X_pca = pca.fit_transform(X_dense)

print("PCA shape:", X_pca.shape)
```

---

## 16. Visualize the K-Means Clusters

```python
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["cluster"],
    cmap="viridis",
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clustering of Publishers")

plt.colorbar(scatter, label="Cluster")

plt.show()
```

---

## 17. Examine Individual Clusters

```python
for cluster in range(k):

    print(f"\nCluster {cluster}")

    cluster_df = df[df["cluster"] == cluster]

    print("Number of publishers:", len(cluster_df))

    display(cluster_df.head(10))
```

---

## 18. Find the Most Representative Publishers

We can identify publishers closest to each cluster centre.

```python
from sklearn.metrics import pairwise_distances

distances = pairwise_distances(
    X,
    kmeans.cluster_centers_,
    metric="cosine"
)

for cluster in range(k):

    cluster_indices = np.where(df["cluster"].values == cluster)[0]

    closest_indices = cluster_indices[
        np.argsort(distances[cluster_indices, cluster])[:10]
    ]

    print(f"\nCluster {cluster} - Representative Publishers")

    display(df.iloc[closest_indices])
```

---

## 19. Save the Results

Save the original dataset with the assigned cluster.

```python
df.to_csv(
    "publishers_clustered.csv",
    index=False
)

print("Saved as publishers_clustered.csv")
```

---

## 20. Create a Cluster Summary

```python
cluster_summary = []

for cluster in range(k):

    cluster_words = [
        terms[ind]
        for ind in order_centroids[cluster, :15]
    ]

    cluster_summary.append({
        "Cluster": cluster,
        "Number_of_Publishers": int(
            (df["cluster"] == cluster).sum()
        ),
        "Top_Words": ", ".join(cluster_words)
    })

cluster_summary = pd.DataFrame(cluster_summary)

display(cluster_summary)
```

Save the summary:

```python
cluster_summary.to_csv(
    "publisher_cluster_summary.csv",
    index=False
)
```

---

# Complete Workflow

The overall process is:

```text
publishers.csv
      ↓
Load dataset
      ↓
Select text columns
      ↓
Combine text
      ↓
Clean text using NLP
      ↓
TF-IDF vectorization
      ↓
Choose number of clusters
      ↓
K-Means
      ↓
Assign cluster to each publisher
      ↓
Analyse important words
      ↓
Visualise clusters
      ↓
Save clustered dataset
```

### Expected Output

Your final dataset will contain an additional column:

```text
publisher | description | category | cluster
------------------------------------------------
Publisher A | ... | Fiction | 0
Publisher B | ... | Academic | 2
Publisher C | ... | Fiction | 0
Publisher D | ... | Children's | 1
```

The `cluster` number itself does **not** have an inherent meaning. The important words and publishers within each cluster should be examined to interpret what the clusters represent.


SyntaxError: invalid decimal literal (4235019594.py, line 332)